<a href="https://colab.research.google.com/github/DaJeong03/text-ai-study/blob/main/text_to_insights_audit_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# 재현성을 위한 시드 고정
np.random.seed(42)
random.seed(42)

def generate_audit_dataset(num_records=1000):
    departments = ["영업1팀", "영업2팀", "마케팅팀", "재무팀", "인사팀", "개발팀", "전략기획팀"]
    dept_weights = [0.25, 0.20, 0.18, 0.08, 0.07, 0.12, 0.10] # 영업/마케팅 비중 높게 설정

    categories = {
        "접대비": {"merchants": ["한우명가", "일식긴자", "청담비스트로", "스카이골프클럽", "로얄와인바"], "min": 100000, "max": 800000},
        "복리후생비": {"merchants": ["스타벅스", "교보문고", "올리브영", "김밥천국", "배달의민족"], "min": 10000, "max": 150000},
        "회의비": {"merchants": ["투썸플레이스", "폴바셋", "본도시락", "도미노피자", "파리바게뜨"], "min": 30000, "max": 200000},
        "여비교통비": {"merchants": ["카카오T택시", "코레일(KTX)", "SK에너지주유소", "GS칼텍스", "하이패스"], "min": 15000, "max": 120000},
        "소모품비": {"merchants": ["알파문구", "쿠팡비즈", "다이소", "오피스디포", "네이버쇼핑"], "min": 20000, "max": 300000},
        "광고선전비": {"merchants": ["메타코리아", "구글애즈", "블로그마케팅대행", "인쇄기획사"], "min": 300000, "max": 2000000},
        "IT유지보수비": {"merchants": ["AWS코리아", "노션랩스", "젯브레인", "가비아"], "min": 50000, "max": 1500000}
    }

    records = []
    start_date = datetime(2026, 1, 1)

    # 1. 일반 거래 및 기본 이상 징후 970건 생성
    for i in range(1, num_records - 29):
        dept = random.choices(departments, weights=dept_weights)[0]

        # 부서별 계정과목 가중치 차등 부여
        if dept in ["영업1팀", "영업2팀"]:
            cat = random.choices(list(categories.keys()), weights=[0.4, 0.15, 0.15, 0.2, 0.05, 0.03, 0.02])[0]
        elif dept == "개발팀":
            cat = random.choices(list(categories.keys()), weights=[0.05, 0.3, 0.2, 0.1, 0.1, 0.05, 0.2])[0]
        else:
            cat = random.choice(list(categories.keys()))

        merchant = random.choice(categories[cat]["merchants"])

        # 날짜 생성 (2026년 상반기: 1월~6월)
        random_days = random.randint(0, 180)
        tx_date = start_date + timedelta(days=random_days)

        # 주말 여부 (5: 토요일, 6: 일요일) -> 의도적으로 약 12%만 주말 결제가 발생하도록 조정
        if tx_date.weekday() >= 5 and random.random() > 0.35:
            tx_date += timedelta(days=random.randint(1, 2))
        is_weekend = tx_date.weekday() >= 5

        # 시간 생성 (일반 업무시간 09~21시 vs 심야 이상 결제 23~04시 약 6% 주입)
        if random.random() < 0.06:
            hour = random.choice([23, 0, 1, 2, 3, 4])
            is_late_night = True
        else:
            hour = random.randint(9, 21)
            is_late_night = False
        minute = random.randint(0, 59)

        # 금액 생성 (1,000원 단위 절사)
        amount = round(random.randint(categories[cat]["min"], categories[cat]["max"]), -3)

        # 접대비 법정 증빙/내부 통제 기준(건당 50만 원) 초과 여부
        is_limit_exceeded = (cat == "접대비" and amount >= 500000) or (amount >= 1000000)

        records.append({
            "tx_id": f"TX-{20260000 + i}",
            "tx_date": tx_date.strftime("%Y-%m-%d"),
            "tx_time": f"{hour:02d}:{minute:02d}",
            "quarter": f"2026-Q{(tx_date.month - 1) // 3 + 1}",
            "month": f"{tx_date.month}월",
            "dept_name": dept,
            "account_category": cat,
            "merchant_name": merchant,
            "amount": amount,
            "is_weekend": is_weekend,
            "is_late_night": is_late_night,
            "is_limit_exceeded": is_limit_exceeded,
            "is_split_suspect": False,
            "audit_risk_type": "정상"
        })

    # 2. 감사 핵심 시나리오: '쪼개기 결제(Split Transaction)' 30건(15쌍) 강제 주입
    # (50만 원 한도 결재를 피하기 위해 같은 날, 같은 가맹점에서 수 분 간격으로 45~49만 원씩 2회 분할 결제)
    for j in range(15):
        base_idx = num_records - 29 + (j * 2)
        split_dept = random.choice(["영업1팀", "영업2팀", "마케팅팀"])
        split_merchant = random.choice(["한우명가", "일식긴자", "로얄와인바"])
        split_date = start_date + timedelta(days=random.randint(0, 180))
        base_hour = random.randint(19, 22)
        base_min = random.randint(10, 45)

        for k in range(2):
            amt = random.choice([450000, 480000, 490000, 495000])
            records.append({
                "tx_id": f"TX-{20260000 + base_idx + k}",
                "tx_date": split_date.strftime("%Y-%m-%d"),
                "tx_time": f"{base_hour:02d}:{base_min + (k * 3):02d}", # 3분 간격 연속 결제
                "quarter": f"2026-Q{(split_date.month - 1) // 3 + 1}",
                "month": f"{split_date.month}월",
                "dept_name": split_dept,
                "account_category": "접대비",
                "merchant_name": split_merchant,
                "amount": amt,
                "is_weekend": split_date.weekday() >= 5,
                "is_late_night": False,
                "is_limit_exceeded": False, # 단일 건으로는 50만 원 미만으로 위장
                "is_split_suspect": True,  # 분할 결제 의심 플래그
                "audit_risk_type": "분할결제(쪼개기) 의심"
            })

    df = pd.DataFrame(records)

    # 3. 감사 위험 유형(audit_risk_type) 라벨링 정교화
    def classify_risk(row):
        if row["is_split_suspect"]:
            return "분할결제(쪼개기) 의심"
        elif row["is_weekend"] and row["is_late_night"]:
            return "주말+심야 중복위험"
        elif row["is_late_night"]:
            return "심야시간 결제(23~04시)"
        elif row["is_weekend"]:
            return "주말/휴일 결제"
        elif row["is_limit_exceeded"]:
            return "고액/한도초과 결제"
        return "정상"

    df["audit_risk_type"] = df.apply(classify_risk, axis=1)

    # 날짜순 정렬 후 CSV 저장
    df = df.sort_values(by=["tx_date", "tx_time"]).reset_index(drop=True)
    df.to_csv("corp_card_audit_1000.csv", index=False, encoding="utf-8-sig")
    print(f"✅ 총 {len(df)}건의 감사 데이터셋 생성 완료! ('corp_card_audit_1000.csv')")
    print("\n[감사 위험 유형별 분포 요약]")
    print(df["audit_risk_type"].value_counts())
    return df

# 실행 및 DuckDB 연동
df_audit = generate_audit_dataset(1000)

✅ 총 1000건의 감사 데이터셋 생성 완료! ('corp_card_audit_1000.csv')

[감사 위험 유형별 분포 요약]
audit_risk_type
정상                 645
고액/한도초과 결제         141
주말/휴일 결제           128
심야시간 결제(23~04시)     52
분할결제(쪼개기) 의심        30
주말+심야 중복위험           4
Name: count, dtype: int64


In [9]:
!pip install -q -U google-genai duckdb gradio plotly pandas

In [13]:
import os
import re
import duckdb
import pandas as pd
import plotly.express as px
import gradio as gr
from google import genai
from google.genai import types
from google.genai.errors import APIError, ServerError

# 1. API 키 설정 (Colab Secrets 우선 확인, 없을 시 환경 변수 확인)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GEMINI_API_KEY", "")

# 빈 키이거나 초기값이면 사용자에게 안내하기 위한 변수
if not api_key or api_key == "YOUR_API_KEY_HERE":
    print("⚠️ 경고: Colab Secrets(열쇠 아이콘)에 'GEMINI_API_KEY'를 추가하시거나 코드에 올바른 API 키를 지정해 주세요.")
    client = None
else:
    client = genai.Client(api_key=api_key)
# 2. DuckDB 설정: 전역 연결(con) 대신 실행 시마다 df_audit을 직접 참조하도록 변경
# (DuckDB는 파이썬 변수에 있는 Pandas DataFrame을 별도 등록 없이 SQL에서 바로 조회할 수 있습니다!)

SCHEMA_INFO = """
테이블명: corp_card_logs (1,000건 규모의 전사 법인카드 지출 및 감사 로그)
컬럼 정보:
- tx_id (VARCHAR): 전표 고유 번호
- tx_date (VARCHAR): 결제 일자 ('YYYY-MM-DD')
- tx_time (VARCHAR): 결제 시각 ('HH:MM')
- quarter (VARCHAR): 분기 ('2026-Q1', '2026-Q2')
- month (VARCHAR): 월 ('1월' ~ '6월')
- dept_name (VARCHAR): 부서명 ('영업1팀', '영업2팀', '마케팅팀', '재무팀', '인사팀', '개발팀', '전략기획팀')
- account_category (VARCHAR): 계정과목 ('접대비', '복리후생비', '회의비', '여비교통비', '소모품비', '광고선전비', 'IT유지보수비')
- merchant_name (VARCHAR): 가맹점명
- amount (INTEGER): 결제 금액 (원)
- is_weekend (BOOLEAN): 주말 결제 여부 (True/False)
- is_late_night (BOOLEAN): 심야(23시~04시) 결제 여부 (True/False)
- is_limit_exceeded (BOOLEAN): 접대비 50만 원 이상 또는 고액 결제 여부 (True/False)
- is_split_suspect (BOOLEAN): 동일 일자/가맹점 연속 분할 결제(쪼개기) 의심 여부 (True/False)
- audit_risk_type (VARCHAR): 감사 위험 분류 ('정상', '주말/휴일 결제', '심야시간 결제(23~04시)', '고액/한도초과 결제', '분할결제(쪼개기) 의심', '주말+심야 중복위험')
"""

# 1단계: Read-Only 보안 가드레일 함수
def validate_sql_readonly(sql: str) -> bool:
    dangerous_keywords = r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE)\b"
    if re.search(dangerous_keywords, sql, re.IGNORECASE):
        return False
    return True

# 2단계: Text-to-SQL 생성 및 Self-Correction 함수 (멀티스레드 충돌 완벽 방지)
def generate_and_execute_sql(user_query: str, max_retries: int = 2):
    if client is None:
        return "", None, "Gemini API Client가 유효하지 않습니다. API 키 설정을 확인해 주세요."

    error_context = ""
    sql_query = ""

    for attempt in range(max_retries + 1):
        prompt = f"""
        당신은 회계 데이터 분석을 위한 DuckDB SQL 전문가입니다.
        아래 스키마를 참고하여 사용자의 질문에 답하는 정확한 SQL 쿼리만 작성하세요.

        [중요 규칙]
        1. 결과는 반드시 막대 그래프(Bar Chart)로 시각화할 수 있도록 **첫 번째 컬럼은 범주형(X축), 두 번째 컬럼은 집계 수치형(Y축)**으로 반드시 2개 이상의 컬럼을 SELECT 하세요.
        2. 마크다운 코드 블록(```sql)이나 부연 설명 없이 오직 실행 가능한 SQL 문장만 출력하세요.
        3. 데이터를 조회하는 SELECT 문만 작성하세요.

        [스키마 정보]
        {SCHEMA_INFO}

        [사용자 질문]
        {user_query}
        {error_context}
        """

        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0)
            )

            sql_query = response.text.strip().replace("```sql", "").replace("```", "").strip()

            # 가드레일 검사 실행
            if not validate_sql_readonly(sql_query):
                return sql_query, None, f"🛡️ [보안 가드레일 차단] 허용되지 않은 데이터 변경 키워드가 감지되었습니다.\n차단된 쿼리: {sql_query}"

            # [핵심 수정] 매 실행마다 독립적인 로컬 DuckDB 세션을 열고 닫아 Gradio 스레드 충돌 원천 차단
            with duckdb.connect(database=':memory:') as local_con:
                local_con.register('corp_card_logs', df_audit)
                result_df = local_con.execute(sql_query).df()

            # 컬럼이 1개만 반환된 경우 Self-Correction 유도
            if len(result_df.columns) < 2:
                raise ValueError("시각화를 위해 최소 2개의 컬럼(X축 범주, Y축 수치)을 SELECT 해야 합니다.")

            return sql_query, result_df, None

        except (ServerError, APIError) as ae:
            if hasattr(ae, 'code') and ae.code in [429, 503]:
                return sql_query, None, f"⚠️ 현재 API 요청 한도 초과 또는 서버 혼잡(코드: {ae.code}) 상태입니다. 약 10~20초 후 다시 시도해 주세요."
            error_context = f"\n[API 오류 발생] {str(ae)}"
        except Exception as e:
            if "503" in str(e) or "429" in str(e) or "temporarily" in str(e).lower():
                return sql_query, None, "⚠️ 현재 Gemini API 서버가 혼잡하거나 분당 호출 한도에 도달했습니다. 잠시 후 다시 요청해 주세요."
            error_context = f"\n[이전 생성 쿼리 오류 발생]\n작성한 쿼리: {sql_query}\n에러 메시지: {str(e)}\n위 에러를 수정하여 올바른 SQL을 다시 작성하세요."

    return sql_query, None, f"최대 재시도 횟수 초과. 마지막 에러: {error_context}"

# 4. 고정 막대 그래프(px.bar) 렌더링 및 회계 인사이트 도출
def analyze_pipeline(user_query: str):
    if not user_query or not user_query.strip():
        return "", None, "질문을 입력해 주세요."

    sql_query, df, error = generate_and_execute_sql(user_query)

    if error or df is None or df.empty:
        return sql_query, None, f"데이터를 추출하지 못했습니다. ({error or '결과 없음'})"

    x_col = df.columns[0]
    y_col = df.columns[1]

    fig = px.bar(
        df, x=x_col, y=y_col, text_auto='.2s',
        title=f"분석 결과: {user_query}",
        labels={x_col: x_col, y_col: y_col},
        color=x_col
    )
    fig.update_layout(showlegend=False, template="plotly_white")

    # [핵심 수정] tabulate 패키지 의존성 에러 방지를 위해 to_markdown() 대신 to_string() 사용
    insight_prompt = f"""
    다음은 '{user_query}'에 대한 DB 조회 결과입니다:
    {df.to_string(index=False)}

    회계법인 데이터 애널리스트 관점에서 위 수치의 핵심 특징과 이상 징후(또는 예산 관리 포인트)를 3줄 이내로 간결하게 요약하세요.
    """
    try:
        insight_resp = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=insight_prompt
        )
        insight_text = insight_resp.text
    except Exception as e:
        insight_text = f"⚠️ 차트는 정상 생성되었으나 AI 인사이트 요약 중 일시적 오류가 발생했습니다: {str(e)}"

    return sql_query, fig, insight_text

# 5. Gradio UI 구동
with gr.Blocks() as demo:
    gr.Markdown("## 📊 재무/회계 데이터 Text-to-Insights 에이전트")
    gr.Markdown("자연어로 질의하면 **SQL 자동 생성(Self-Correction) → DuckDB 실행 → 막대 차트 시각화 → 회계 감사 인사이트**를 도출합니다.")

    with gr.Row():
        query_input = gr.Textbox(
            label="자연어 분석 질의",
            placeholder="예: 부서별 총 지출 금액을 비교해줘 / 주말에 결제된 계정과목별 금액 합계를 보여줘",
            scale=4
        )
        submit_btn = gr.Button("분석 실행", variant="primary", scale=1)

    gr.Examples(
        examples=[
            ["부서별 법인카드 총 지출 금액 상위 순으로 보여줘"],
            ["주말(is_weekend가 True)에 결제된 내역을 부서별로 합산해서 보여줘"],
            ["계정과목별 평균 지출 금액을 비교해줘"]
        ],
        inputs=query_input
    )

    with gr.Row():
        sql_output = gr.Code(label="생성 및 검증된 DuckDB SQL", language="sql")
        insight_output = gr.Markdown(label="AI 회계/재무 분석 리포트")

    chart_output = gr.Plot(label="시각화 결과 (Bar Chart)")

    submit_btn.click(
        fn=analyze_pipeline,
        inputs=query_input,
        outputs=[sql_output, chart_output, insight_output]
    )

# Gradio 6.0 스펙에 맞춰 Blocks 생성자에서 빼고 launch 내부에 theme 설정 부여
demo.launch(theme=gr.themes.Soft(), debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2b4aa20d8eef387531.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e592ce7c7dd6f4de9c.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://2b4aa20d8eef387531.gradio.live
